In [1]:
import warnings

import keras
import xgboost as xgb
import joblib
from service.datasetservice.DatasetService import DatasetService
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=UserWarning)

%load_ext autoreload
%autoreload 2

# Shallow NN

## Centralized With Fine Tuning

In [2]:
dataset_service = DatasetService()

x_all, y_all = dataset_service.get_subject_data(subject="all", with_features=True)
x_train, x_test, y_train, y_test = dataset_service.train_test_split(x=x_all, y=y_all)

model: keras.Sequential = joblib.load(f"shallow_nn.pkl")

early_stopping_callback = keras.callbacks.EarlyStopping(patience=10, monitor="val_loss", min_delta=0.001)
reduce_lr_callback = keras.callbacks.ReduceLROnPlateau(patience=7, monitor="val_loss", factor=0.2)
model.fit(x_train, y_train, epochs=150, batch_size=32, verbose=2, validation_split=0.2, callbacks=[early_stopping_callback, reduce_lr_callback])

y_pred = model.predict(x_test, verbose=False)
f1 = f1_score(y_true=y_test, y_pred=y_pred > 0.5)
acc = accuracy_score(y_true=y_test, y_pred=y_pred > 0.5)
print(f"F1", f1)
print(f"Accuracy", acc)

Epoch 1/150
1970/1970 - 6s - 3ms/step - accuracy: 0.6849 - loss: 0.5933 - val_accuracy: 0.6934 - val_loss: 0.5736 - learning_rate: 0.0010
Epoch 2/150
1970/1970 - 5s - 3ms/step - accuracy: 0.6902 - loss: 0.5774 - val_accuracy: 0.7032 - val_loss: 0.5549 - learning_rate: 0.0010
Epoch 3/150
1970/1970 - 6s - 3ms/step - accuracy: 0.6940 - loss: 0.5650 - val_accuracy: 0.7078 - val_loss: 0.5438 - learning_rate: 0.0010
Epoch 4/150
1970/1970 - 5s - 3ms/step - accuracy: 0.7007 - loss: 0.5544 - val_accuracy: 0.6865 - val_loss: 0.5651 - learning_rate: 0.0010
Epoch 5/150
1970/1970 - 5s - 3ms/step - accuracy: 0.7049 - loss: 0.5439 - val_accuracy: 0.7108 - val_loss: 0.5335 - learning_rate: 0.0010
Epoch 6/150
1970/1970 - 5s - 3ms/step - accuracy: 0.7145 - loss: 0.5302 - val_accuracy: 0.7275 - val_loss: 0.5112 - learning_rate: 0.0010
Epoch 7/150
1970/1970 - 5s - 3ms/step - accuracy: 0.7226 - loss: 0.5200 - val_accuracy: 0.7402 - val_loss: 0.5027 - learning_rate: 0.0010
Epoch 8/150
1970/1970 - 5s - 3ms/s

## Centralized Without Fine Tuning

In [12]:
dataset_service = DatasetService()

x_all, y_all = dataset_service.get_subject_data(subject="all", with_features=True)
x_train, x_test, y_train, y_test = dataset_service.train_test_split(x=x_all, y=y_all)

model: keras.Sequential = joblib.load(f"shallow_nn.pkl")

y_pred = model.predict(x_test, verbose=False)
f1 = f1_score(y_true=y_test, y_pred=y_pred > 0.5)
acc = accuracy_score(y_true=y_test, y_pred=y_pred > 0.5)
print(f"F1", f1)
print(f"Accuracy", acc)

F1 0.4806171648987464
Accuracy 0.3163239400863163


## Individual Fine Tuned

In [18]:
dataset_service = DatasetService()

for subject in range(2, 36):
    x, y = dataset_service.get_subject_data(subject=subject, with_features=True)
    x_train, x_test, y_train, y_test = dataset_service.train_test_split(x=x, y=y)

    model: keras.Sequential = joblib.load(f"shallow_nn.pkl")

    early_stopping_callback = keras.callbacks.EarlyStopping(patience=10, monitor="val_loss", min_delta=0.001)
    reduce_lr_callback = keras.callbacks.ReduceLROnPlateau(patience=7, monitor="val_loss", factor=0.2)
    model.fit(x_train, y_train, epochs=150, batch_size=32, verbose=False, validation_split=0.2, callbacks=[early_stopping_callback, reduce_lr_callback])

    y_pred = model.predict(x_test, verbose=False)
    f1 = f1_score(y_true=y_test, y_pred=y_pred > 0.5)
    acc = accuracy_score(y_true=y_test, y_pred=y_pred > 0.5)
    print(f"Subject {subject} F1: ", f1)
    print(f"Subject {subject} Accuracy", acc)

Subject 2 F1:  0.9948979591836735
Subject 2 Accuracy 0.9968203497615262
Subject 3 F1:  0.9936708860759493
Subject 3 Accuracy 0.996551724137931
Subject 4 F1:  0.9915966386554622
Subject 4 Accuracy 0.9951923076923077
Subject 5 F1:  0.9946808510638298
Subject 5 Accuracy 0.9965034965034965
Subject 6 F1:  1.0
Subject 6 Accuracy 1.0
Subject 7 F1:  0.996996996996997
Subject 7 Accuracy 0.9983108108108109
Subject 8 F1:  1.0
Subject 8 Accuracy 1.0
Subject 9 F1:  0.9967845659163987
Subject 9 Accuracy 0.9981916817359855
Subject 10 F1:  1.0
Subject 10 Accuracy 1.0
Subject 11 F1:  0.9975308641975309
Subject 11 Accuracy 0.9982517482517482
Subject 12 F1:  1.0
Subject 12 Accuracy 1.0
Subject 13 F1:  0.9949748743718593
Subject 13 Accuracy 0.9965811965811966
Subject 14 F1:  0.995
Subject 14 Accuracy 0.9968304278922345
Subject 15 F1:  0.9973045822102425
Subject 15 Accuracy 0.9983552631578947
Subject 16 F1:  0.996996996996997
Subject 16 Accuracy 0.9983471074380166
Subject 17 F1:  0.9972451790633609
Subject

## Individual Without Fine Tuning

In [19]:
dataset_service = DatasetService()

for subject in range(2, 36):
    x, y = dataset_service.get_subject_data(subject=subject, with_features=True)
    x_train, x_test, y_train, y_test = dataset_service.train_test_split(x=x, y=y)

    model: keras.Sequential = joblib.load(f"shallow_nn.pkl")

    y_pred = model.predict(x_test, verbose=False)
    f1 = f1_score(y_true=y_test, y_pred=y_pred > 0.5)
    acc = accuracy_score(y_true=y_test, y_pred=y_pred > 0.5)
    print(f"Subject {subject} F1: ", f1)
    print(f"Subject {subject} Accuracy", acc)

Subject 2 F1:  0.6416184971098265
Subject 2 Accuracy 0.8028616852146264
Subject 3 F1:  0.3076923076923077
Subject 3 Accuracy 0.7672413793103449
Subject 4 F1:  0.5836575875486382
Subject 4 Accuracy 0.8285256410256411
Subject 5 F1:  0.5023474178403756
Subject 5 Accuracy 0.6293706293706294
Subject 6 F1:  0.26905829596412556
Subject 6 Accuracy 0.7189655172413794
Subject 7 F1:  0.022988505747126436
Subject 7 Accuracy 0.7128378378378378
Subject 8 F1:  0.6989010989010989
Subject 8 Accuracy 0.7486238532110092
Subject 9 F1:  0.46540880503144655
Subject 9 Accuracy 0.538878842676311
Subject 10 F1:  0.6015831134564644
Subject 10 Accuracy 0.7118320610687023
Subject 11 F1:  0.5521739130434783
Subject 11 Accuracy 0.6398601398601399
Subject 12 F1:  0.10204081632653061
Subject 12 Accuracy 0.6986301369863014
Subject 13 F1:  0.0
Subject 13 Accuracy 0.6615384615384615
Subject 14 F1:  0.5171503957783641
Subject 14 Accuracy 0.7099841521394612
Subject 15 F1:  0.3944954128440367
Subject 15 Accuracy 0.56578947

# XGBoost

## Centralized With Fine Tuning

In [25]:
dataset_service = DatasetService()

x_all, y_all = dataset_service.get_subject_data(subject="all", with_features=True)
x_train, x_test, y_train, y_test = dataset_service.train_test_split(x=x_all, y=y_all)

bst: xgb.Booster = joblib.load(f"xgboost.pkl")
params = joblib.load("params.pkl")

# Local training
bst = xgb.train(params, xgb.DMatrix(x_train, label=y_train), num_boost_round=1000, xgb_model=bst, early_stopping_rounds=10, evals=[(xgb.DMatrix(x_test, label=y_test), "valid")])

y_pred = bst.predict(xgb.DMatrix(x_test, label=y_test))
f1 = f1_score(y_true=y_test, y_pred=y_pred > 0.5)
acc = accuracy_score(y_true=y_test, y_pred=y_pred > 0.5)
print(f"F1", f1)
print(f"Accuracy", acc)

[0]	valid-auc:0.67995
[1]	valid-auc:0.70142
[2]	valid-auc:0.72069
[3]	valid-auc:0.73803
[4]	valid-auc:0.75277
[5]	valid-auc:0.76631
[6]	valid-auc:0.77818
[7]	valid-auc:0.78681
[8]	valid-auc:0.79371
[9]	valid-auc:0.80206
[10]	valid-auc:0.80895
[11]	valid-auc:0.81744
[12]	valid-auc:0.82245
[13]	valid-auc:0.82780
[14]	valid-auc:0.83340
[15]	valid-auc:0.84033
[16]	valid-auc:0.84368
[17]	valid-auc:0.84804
[18]	valid-auc:0.85100
[19]	valid-auc:0.85572
[20]	valid-auc:0.86254
[21]	valid-auc:0.86720
[22]	valid-auc:0.87181
[23]	valid-auc:0.87442
[24]	valid-auc:0.87779
[25]	valid-auc:0.88177
[26]	valid-auc:0.88379
[27]	valid-auc:0.88590
[28]	valid-auc:0.88891
[29]	valid-auc:0.89299
[30]	valid-auc:0.89667
[31]	valid-auc:0.89914
[32]	valid-auc:0.90190
[33]	valid-auc:0.90542
[34]	valid-auc:0.91053
[35]	valid-auc:0.91325
[36]	valid-auc:0.91671
[37]	valid-auc:0.91797
[38]	valid-auc:0.92077
[39]	valid-auc:0.92403
[40]	valid-auc:0.92559
[41]	valid-auc:0.92867
[42]	valid-auc:0.93235
[43]	valid-auc:0.9336

## Centralized Without Fine Tuning

In [26]:
dataset_service = DatasetService()

x_all, y_all = dataset_service.get_subject_data(subject="all", with_features=True)
x_train, x_test, y_train, y_test = dataset_service.train_test_split(x=x_all, y=y_all)

bst: xgb.Booster = joblib.load(f"xgboost.pkl")

y_pred = bst.predict(xgb.DMatrix(x_test, label=y_test))
f1 = f1_score(y_true=y_test, y_pred=y_pred > 0.5)
acc = accuracy_score(y_true=y_test, y_pred=y_pred > 0.5)
print(f"F1", f1)
print(f"Accuracy", acc)

F1 0.43697072344543914
Accuracy 0.6455445544554456


## XGBoost Individual With Fine Tuning

In [30]:
dataset_service = DatasetService()

for subject in range(2, 36):
    x, y = dataset_service.get_subject_data(subject=subject, with_features=True)
    x_train, x_test, y_train, y_test = dataset_service.train_test_split(x=x, y=y)
    x, y = dataset_service.get_subject_data(subject="all", with_features=True)
    x_train_all, x_test_all, y_train_all, y_test_all = dataset_service.train_test_split(x=x, y=y)

    bst: xgb.Booster = joblib.load(f"xgboost.pkl")
    params = joblib.load("params.pkl")

    # Local training
    bst = xgb.train(params, xgb.DMatrix(x_train, label=y_train), num_boost_round=1000, xgb_model=bst, early_stopping_rounds=10, evals=[(xgb.DMatrix(x_test, label=y_test), "valid")], verbose_eval=False)

    y_pred = bst.predict(xgb.DMatrix(x_test, label=y_test))
    f1 = f1_score(y_true=y_test, y_pred=y_pred > 0.5)
    acc = accuracy_score(y_true=y_test, y_pred=y_pred > 0.5)
    print(f"Subject {subject} F1: ", f1)
    print(f"Subject {subject} Accuracy", acc)

Subject 2 F1:  0.9948979591836735
Subject 2 Accuracy 0.9968203497615262
Subject 3 F1:  0.9936305732484076
Subject 3 Accuracy 0.996551724137931
Subject 4 F1:  0.9972144846796658
Subject 4 Accuracy 0.9983974358974359
Subject 5 F1:  1.0
Subject 5 Accuracy 1.0
Subject 6 F1:  1.0
Subject 6 Accuracy 1.0
Subject 7 F1:  0.9940476190476191
Subject 7 Accuracy 0.9966216216216216
Subject 8 F1:  0.996996996996997
Subject 8 Accuracy 0.998165137614679
Subject 9 F1:  0.9967845659163987
Subject 9 Accuracy 0.9981916817359855
Subject 10 F1:  0.9969788519637462
Subject 10 Accuracy 0.9980916030534351
Subject 11 F1:  1.0
Subject 11 Accuracy 1.0
Subject 12 F1:  1.0
Subject 12 Accuracy 1.0
Subject 13 F1:  0.9974811083123426
Subject 13 Accuracy 0.9982905982905983
Subject 14 F1:  0.9975062344139651
Subject 14 Accuracy 0.9984152139461173
Subject 15 F1:  0.9973190348525469
Subject 15 Accuracy 0.9983552631578947
Subject 16 F1:  1.0
Subject 16 Accuracy 1.0
Subject 17 F1:  1.0
Subject 17 Accuracy 1.0
Subject 18 F1: 